# Classifying Documents

# **Classifying Document**

Estimated time needed: **60** minutes

Imagine working at a prestigious newspaper or magazine company that boasts an extensive archive of documents dating back through the annals of time. Amid this treasure trove of information, a monumental task lies ahead: organizing these historical documents into their relevant topic sections. This strategic curation not only promises to enhance the user experience by delivering more streamlined content but also presents an opportunity to breathe new life into invaluable insights from the past through a modern lens. However, the sheer volume and scope of this undertaking call for a sophisticated solution. 

![Documents Overload](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-GPXX0Y15EN/docs.png)

The implementation of an automated machine learning system makes it very efficient. Such a system, equipped with advanced natural language processing and machine learning capabilities, could sift through the vast archives, categorizing articles into their respective topics with remarkable precision. As a result, readers would seamlessly access a wealth of knowledge tailored to their interests, while the editorial team gains newfound agility in content management.

In this project, you will embark on the exciting task of classifying news articles for a content search engine. The goal is to build a model that can automatically categorize news articles into different topics or classes, enabling the search engine to deliver relevant content to users efficiently. To achieve this, you will leverage the powerful torchtext library, which simplifies the process of creating a dataset for text classification analysis.

With torchtext, you'll have the flexibility to access and preprocess raw news data effortlessly. The library enables you to convert text strings into torch.Tensors, which are essential for training machine learning models. By using torchtext's convenient functionalities, you can set up an efficient data processing pipeline that prepares your text data for classification.

Throughout this tutorial, you'll demonstrate how to effectively shuffle and iterate through the processed data using torch.utils.data.DataLoader. This DataLoader simplifies the data handling process, allowing you to focus on building and training your text classification model effectively.


# Objectives

After completing this lab, you will be able to:

- Work with datasets and understand tokenizer, embedding bag technique and vocabulary.
- Explore embeddings in PyTorch and understand token indices.
- Perform text classification using data loader and apply it on a neural network model.
- Train the text classification model on a news dataset.
- Engage in various exercises to solidify your understanding.


## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import torch
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from torch.utils.data import DataLoader

## Text classification
Let's build a text classification model using PyTorch and torchtext to classify news articles into one of the four categories: World, Sports, Business, and Sci/Tech.


### Import bank dataset

Load the AG_NEWS dataset for the train split and split it into input text and corresponding labels:


In [26]:
from torchtext.datasets import AG_NEWS

In [32]:
train_iter = iter(AG_NEWS(split="train")) # iterator object

The AG_NEWS dataset in torchtext does not support direct indexing like a list or tuple. It is not a random access dataset but rather an iterable dataset that needs to be used with an iterator. This approach is more effective for text data.


In [33]:
print(train_iter)

<generator object ShardingFilterIterDataPipe.__iter__ at 0x00000289368A10C0>


In [ ]:
(y,text) = train_iter.__next__()
# seems torch dataset has issues beacuse of poorly updation since 2023
# So we move to hugging face dataset  

AttributeError: 'NoneType' object has no attribute 'Lock'
This exception is thrown by __iter__ of _MemoryCellIterDataPipe(remember_elements=1000, source_datapipe=_ChildDataPipe)

In [106]:
from datasets import load_dataset

In [107]:
dataset = load_dataset("AG_NEWS")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [108]:
train_iter = iter(dataset["train"])

In [109]:
sample = next(train_iter)
print("Label:",sample["label"])
print("Text:", sample["text"])

Label: 2
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


Find the label of the sample.


In [110]:
ag_news_label = {1:"World", 2:"Sports", 3:"Business", 4:"Sci/Tec"}
ag_news_label[sample["label"]]


'Sports'

Also, use the dataset to find all the classes.


In [111]:
num_classes = len(set([item["label"] for item in train_iter]))
num_classes

4

### Tokenization

Create the tokens, also build the vocabulary as before, just using the AG dataset to obtain token indices


In [ ]:
tokenizer = get_tokenizer("basic_english")
dataset = load_dataset("AG_NEWS")
# define function for tokenize the iteration
#Hugging face dataset iterator have dict type data 
def yield_token(dataset):
    for row in dataset:
        yield tokenizer(row["text"].lower())
        
# Build vocabulary
vocab = build_vocab_from_iterator(yield_token(dataset["train"]),specials= ["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [123]:
print("Vocab Size:",len(vocab))
print("sample tokens:", list(vocab.get_stoi().keys())[:10])

Vocab Size: 95811
sample tokens: ['television', 'misnomer', 'television-watching', 'new', 'costs', '7-week', 'dutch', '35-10', 'tarceva', 'confidence']


### Dataset

You can convert the dataset into map-style datasets and then perform a random split to create separate training and validation datasets. The training dataset will contain 95% of the samples, while the validation dataset will contain the remaining 5%. These datasets can be used for training and evaluating a machine learning model for text classification on the AG_NEWS dataset.


In [ ]:
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import random_split,Dataset


In [153]:
dataset = load_dataset("ag_news")

# HuggingFace dataset is ALREADY map-style — no conversion needed
# Just wrap it in a PyTorch Dataset to return (label, text) tuples
class AGNewsDataset(Dataset):
    
    def __init__(self, hf_split):
        self.data = hf_split
    
    def __len__(self):
        return len(self.data) 
    
    def __getitem__(self, index):
        row = self.data[index]
        return row["label"], row["text"]
    
train_dataset = AGNewsDataset(dataset["train"])
test_dataset = AGNewsDataset(dataset["test"])

num_train = int(len(train_dataset) * 0.95)
num_valid = len(train_dataset) - num_train

split_train, split_valid = random_split(train_dataset, [num_train, num_valid])
print("Train size:", num_train)
print("Valid size:", num_valid)
print("Test size:", len(test_dataset))

Train size: 114000
Valid size: 6000
Test size: 7600


In [144]:
split_train[6]

(2,
 'Sun Drops the Gloves Sun Microsystems (Nasdaq: SUNW) shareholders have one less headache to medicate thanks to management #39;s wise decision to settle its patent infringement litigation with crusty old Eastman Kodak (NYSE: EK).')

The code checks if a CUDA-compatible GPU is available in the system using PyTorch, a popular deep learning framework. If a GPU is available, it assigns the device variable to "cuda" (which stands for CUDA, the parallel computing platform and application programming interface model developed by NVIDIA). If a GPU is not available, it assigns the device variable to "cpu" (which means the code will run on the CPU instead).


In [154]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [156]:
device.type

'cpu'